# MobileNetV2 --- magnitude at sparsity 0.995

Fills the one hole in MobileNetV2's grid. Magnitude was swept before 0.995
was adopted, so it covers 0.5/0.95/0.97/0.99/0.999 while SNIP-it and WANDA
cover 0.95/0.97/0.99/0.995. Six cells, three seeds, both arms.

## Why this cell decides something

At 0.995 the three criteria separate hardest. SNIP-it gives its largest
margin there (I.P. 61.13 vs BaCP 74.43, +13.30) while WANDA is dead for both
arms. Magnitude's margin has been *narrowing* --- +3.92, +3.04, +1.71 at
0.95/0.97/0.99 --- and the paper currently says so, contrasting it with
SNIP-it's widening margin. That contrast rests on magnitude stopping at 0.99.

If magnitude at 0.995 keeps narrowing toward zero, the contrast holds. If it
turns and widens like SNIP-it, the paragraph in `04_results.tex` needs
rewriting, because the difference would be about *when* each criterion breaks
rather than about a genuine ordering between them.

## Protocol

Both arms at SGD 0.1, the finalised MobileNetV2 setting: this model inherited
the ResNet recipe untuned, and 0.01 collapses its I.P. arm at high sparsity
(chance at 0.99, against 78.63 under 0.1). BaCP's default already is 0.1.

Records use main keys, matching how the SNIP/WANDA I.P. cells were written.
Note that magnitude's *earlier* I.P. cells at 0.5/0.95/0.97/0.99/0.999 sit
under `.lr0.1` variant keys instead, because at the time 0.1 was still being
treated as a deviation from the family default. Same configuration either
way; the key namespaces simply differ.


In [ ]:
import sys, pathlib
here = pathlib.Path.cwd()
for cand in [here, *here.parents]:
    if (cand / 'nb_common.py').exists():
        sys.path.insert(0, str(cand)); break
    if (cand / 'project' / 'test_notebooks' / 'nb_common.py').exists():
        sys.path.insert(0, str(cand / 'project' / 'test_notebooks')); break
else:
    raise RuntimeError('cannot find nb_common.py -- start the kernel inside the repo')
import nb_common as nb
info = nb.setup()

## Preflight

In [ ]:
MODEL, GPU, SEEDS, SP = 'mobilenet_v2', 0, (1, 2, 3), 0.995
nb.fetch_imagenet_weights(MODEL)
nb.preflight(MODEL, num_classes=nb.FAMILIES[MODEL]['base']['num_classes'])

## Dense baselines

Already recorded for all three seeds, so these skip. Present so the notebook
stands alone if run on a fresh results directory.

In [ ]:
nb.run_group([nb.make_cell(MODEL, 'dense', seed=s) for s in SEEDS], gpu=GPU)

## Run --- six cells, arms interleaved per seed

In [ ]:
plan = []
for seed in SEEDS:
    plan.append(nb.make_cell(MODEL, 'prune', seed=seed, pruner='magnitude',
                             sparsity=SP, learning_rate=0.1))
    plan.append(nb.make_cell(MODEL, 'bacp', seed=seed, pruner='magnitude',
                             sparsity=SP))

for c in plan:
    assert c['config']['learning_rate'] == 0.1, c['config']['learning_rate']
    assert c['config']['target_sparsity'] == SP, c['config']['target_sparsity']
    print(f"  {c['key']:64s} lr={c['config']['learning_rate']}")

assert nb.sanity_check(plan), 'sanity check failed'
nb.run_group(plan, gpu=GPU)

## Verdict --- does magnitude keep narrowing?

In [ ]:
import json, glob, os, statistics as st
root = os.environ['BACP_RESULTS_DIR']
acc = {}
for f in glob.glob(os.path.join(root, 'runs', '*.json')):
    r = json.load(open(f, encoding='utf-8'))
    k = r.get('experiment_group') or ''
    if r.get('status') == 'ok' and '.smoke' not in k and 'mobilenet' in k:
        acc[k] = r.get('test_acc_exact_pct') or r.get('test_acc_pct')

def grab(arm, sp, variant=''):
    xs = []
    for n in SEEDS:
        key = f'static.{arm}.{MODEL}.cifar10.s{sp}.magnitude.seed{n}{variant}'
        v = acc.get(key)
        if v is not None:
            xs.append(v)
    if not xs:
        return None
    return st.mean(xs), (st.stdev(xs) if len(xs) > 1 else 0.0), len(xs)

print(f'{"sparsity":>9} | {"I.P.":>16} | {"BaCP":>16} | {"delta":>7}')
prev = None
for sp, var in ((0.95, '.lr0.1'), (0.97, '.lr0.1'), (0.99, '.lr0.1'), (0.995, '')):
    a = grab('prune', sp, var)
    b = grab('bacp', sp)
    fa = f'{a[0]:.2f}+-{a[1]:.2f}(n={a[2]})' if a else '       --       '
    fb = f'{b[0]:.2f}+-{b[1]:.2f}(n={b[2]})' if b else '       --       '
    d = (b[0] - a[0]) if (a and b) else None
    print(f'{sp:>9} | {fa:>16} | {fb:>16} | {d:>+7.2f}' if d is not None
          else f'{sp:>9} | {fa:>16} | {fb:>16} |      --')
    if sp == 0.995 and d is not None and prev is not None:
        print()
        if d > prev:
            print(f'magnitude TURNS AND WIDENS at 0.995 ({prev:+.2f} -> {d:+.2f}).')
            print('The narrowing-vs-widening contrast in 04_results.tex needs rewriting.')
        else:
            print(f'magnitude KEEPS NARROWING at 0.995 ({prev:+.2f} -> {d:+.2f}).')
            print('The contrast with SNIP-it holds as written.')
    if d is not None:
        prev = d